# SymbolicMLP Demo

In this demo, we show you how to:
* Wrap an MLP block in a PyTorch model with our `SymbolicModel` class
* Perform symbolic regresson on the MLP with the `distill` method 
* Switch the MLP to an equation in the forward pass of the model with the `switch_to_equation` method 
* Switch back to the MLP with the `switch_to_block` method

## Wrapping a PyTorch model
Create a simple PyTorch model.

In [4]:
import torch
import numpy as np
import torch.nn as nn

class MLP(nn.Module):
    """
    Simple MLP.
    """
    def __init__(self, input_dim, output_dim, hidden_dim):
        super(MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.mlp(x)

class SimpleModel(nn.Module):
    """
    Simple model class.
    """
    def __init__(self, input_dim, output_dim, hidden_dim = 64):
        super(SimpleModel, self).__init__()

        self.mlp = MLP(input_dim, output_dim, hidden_dim)

    def forward(self, x):
        x = self.mlp(x)
        return x

Train the model on some data.

$$
y = x_0^2 +3 \sin(x_4)-4
$$

In [5]:
# Make the dataset 
x = np.array([np.random.uniform(0, 1, 10_000) for _ in range(5)]).T  
y = x[:, 0]**2 + 3*np.sin(x[:, 4]) - 4
noise = np.array([np.random.normal(0, 0.05*np.std(y)) for _ in range(len(y))])
y = y + noise 

In [6]:
# Set up training

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

def train_model(model, dataloader, opt, criterion, epochs = 100):
    """
    Train a model for the specified number of epochs.
    
    Args:
        model: PyTorch model to train
        dataloader: DataLoader for training data
        opt: Optimizer
        criterion: Loss function
        epochs: Number of training epochs
        
    Returns:
        tuple: (trained_model, loss_tracker)
    """
    loss_tracker = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_x, batch_y in dataloader:
            # Forward pass
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            
            # Backward pass
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            epoch_loss += loss.item()
        
        loss_tracker.append(epoch_loss)
        if (epoch + 1) % 5 == 0:
            avg_loss = epoch_loss / len(dataloader)
            print(f'Epoch [{epoch+1}/{epochs}], Avg Loss: {avg_loss:.6f}')
    return model, loss_tracker

# Instantiate the model
model = SimpleModel(input_dim=x.shape[1], output_dim=1)

# Set up training
criterion = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=0.001)
X_train, _, y_train, _ = train_test_split(x, y.reshape(-1,1), test_size=0.2, random_state=290402)

# Set up dataset
dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [7]:
# Train the model and save the weights

model, losses = train_model(model, dataloader, opt, criterion, 20)
torch.save(model.state_dict(), 'model_weights.pth')

Epoch [5/20], Avg Loss: 0.090289
Epoch [10/20], Avg Loss: 0.065790
Epoch [15/20], Avg Loss: 0.052848
Epoch [20/20], Avg Loss: 0.042653


Wrap the mlp layer in the trained model with SymbolicMLP.

In [ ]:
from symtorch import SymbolicModel
model.mlp = SymbolicModel(model.mlp, block_name = 'Sequential')

## Interpret the MLP

In this example, we pass extra parameters into the `.distill` method (complexity of operators/constants and parsimony, which is a penalisation of complexity).\
To see all the possible parameters, please see the `PySRRegressor` class from [PySR](https://ai.damtp.cam.ac.uk/pysr/api/).

In this example, we turn verbosity off because we are in a Jupyter notebook. For best performance, run in IPython, as you can terminate the SR any time.

In [10]:
# Configure the SR

sr_params = {'complexity_of_operators':  {"sin":3, "exp":3},
             'complexity_of_constants': 2, 
             'parsimony': 0.1,
             'verbosity': 0,
             'niterations': 500}

model.mlp.distill(torch.FloatTensor(X_train),
                       sr_params = sr_params, 
                       parent_model=model) #Pass in the parent model (really only required if the MLP is 
                                            #not at the start of the model but it is good practice)

🛠️ Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


💡Best equation for output 0 found to be ((sin(x4) * 2.8699453) + (x0 * sin(x0))) + -3.9008763.
❤️ SR on Sequential complete.


{0: PySRRegressor.equations_ = [
 	    pick     score                                           equation  \
 	0         0.000000                                                 x4   
 	1         2.630882                                         -2.2768116   
 	2         0.412233                                    x4 + -2.7789853   
 	3         0.489161                             x4 + (x4 + -3.2811875)   
 	4         0.195020                       (x4 + -1.431319) * 2.4504333   
 	5         1.052015                      ((x4 + x0) + x4) + -3.7788339   
 	6         0.932839                 (x0 + (x4 * 2.446758)) + -4.003188   
 	7         0.128709          (x0 * x0) + ((x4 * 2.447426) + -3.836864)   
 	8         0.368714            (x0 + -4.098843) + (sin(x4) * 2.868503)   
 	9         0.266736   ((x0 * x0) + -3.9324567) + (sin(x4) * 2.8690972)   
 	10        0.032179  (x4 * ((x4 * -0.9658124) + 3.4169724)) + ((x0 ...   
 	11  >>>>  0.366639  ((sin(x4) * 2.8699453) + (x0 * sin(x0))) + -3

See the full Pareto front of equations. The best equation is chosen as a balance of accuracy and complexity.\
Outputs from *PySR* are saved in `SR_output/MLP_name`.

In [13]:
model.mlp.show_symbolic_expression()


➡️ Symbolic expressions for output dimension 0:
    complexity      loss                                           equation  \
0            1  7.974565                                                 x4   
1            2  0.574288                                         -2.2768116   
2            4  0.251808                                    x4 + -2.7789853   
3            6  0.094665                             x4 + (x4 + -3.2811875)   
4            7  0.077892                       (x4 + -1.431319) * 2.4504333   
5            8  0.027202                      ((x4 + x0) + x4) + -3.7788339   
6            9  0.010702                 (x0 + (x4 * 2.446758)) + -4.003188   
7           11  0.008273          (x0 * x0) + ((x4 * 2.447426) + -3.836864)   
8           12  0.005722            (x0 + -4.098843) + (sin(x4) * 2.868503)   
9           14  0.003356   ((x0 * x0) + -3.9324567) + (sin(x4) * 2.8690972)   
10          16  0.003147  (x4 * ((x4 * -0.9658124) + 3.4169724)) + ((x0 ...   
11 

## Switch to Using the Equation Instead in the Forwards Pass

You can choose the equation you want to switch to by choosing the desired complexity of equation. \
If left blank, then we choose the best equation chosen by *PySR*.

In [14]:
model.mlp.switch_to_symbolic(complexity=[14]) 

✅ Successfully switched Sequential to symbolic equations for all 1 dimensions:
   Dimension 0: ((x0 * x0) + -3.9324567) + (sin(x4) * 2.8690972)
   Variables: ['x0', 'x4']
🎯 All 1 output dimensions now using symbolic equations.


Now when running the forwards pass through the model, it uses the symbolic equation instead of the MLP. 

In [15]:
interpretable_outputs = model(torch.tensor(X_train, dtype=torch.float32))
interpretable_outputs

tensor([[-3.5793],
        [-3.0994],
        [-1.3390],
        ...,
        [-1.9961],
        [-1.5402],
        [-1.6144]])

You can also make a callable function using the `get_symbolic_function` method.

In [19]:
symbolic_function = model.mlp.get_symbolic_function(complexity = 14)
symbolic_function(X_train)

array([-3.5793202, -3.0994322, -1.3390288, ..., -1.9960518, -1.5401933,
       -1.6143596], shape=(8000,), dtype=float32)

## Switch to Using the MLP in the Forwards Pass

In [20]:
mlp_outputs = model.mlp.switch_to_block()

✅ Switched Sequential back to block


In [21]:
model.eval()
with torch.no_grad():
    model_outputs = model.mlp(torch.tensor(X_train, dtype=torch.float32))
model_outputs

tensor([[-3.5154],
        [-3.2639],
        [-1.2957],
        ...,
        [-1.9459],
        [-1.5621],
        [-1.5987]])

## Wrapping a Python Function

`SymbolicModel` also works with generic Python functions

In [23]:
# Same function as before
def f(X):
    return X[:, 0]**2 + 3*np.sin(X[:, 4]) - 4

In [ ]:
symbolic_function = SymbolicModel(f, block_name = "callable_function")

In [ ]:
symbolic_function.distill(X_train, sr_params=sr_params)

🛠️ Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


💡Best equation for output 0 found to be ((sin(x4) * 3.0) + (x0 * (x0 + -6.008807e-8))) + -4.0.
❤️ SR on callable_function complete.


{0: PySRRegressor.equations_ = [
 	   pick     score                                           equation  \
 	0        0.000000                                                 x4   
 	1        2.549633                                         -2.3011773   
 	2        0.385374                                    x4 + -2.7961826   
 	3        0.457093                             (x4 + -3.2911794) + x4   
 	4        0.254146                       (x4 * 2.5656352) + -3.571168   
 	5        0.943306                      (x4 + (x0 + x4)) + -3.7896304   
 	6        1.421161                (x0 + (x4 * 2.5715055)) + -4.072485   
 	7        0.562942            (x0 * x0) + ((x4 * 2.5713) + -3.905886)   
 	8        4.351141   (-4.0000057 + (x0 * x0)) + (3.0001585 * sin(x4))   
 	9  >>>>  4.305316  ((sin(x4) * 3.0) + (x0 * (x0 + -6.008807e-8)))...   
 	
 	           loss  complexity  
 	0  8.111766e+00           1  
 	1  6.336129e-01           2  
 	2  2.931519e-01           4  
 	3  1.175078e-01     

**Note:** if distilling a generic function (not a PyTorch MLP) you cannot pass in the `parent_model` parameter. If this function is part of a hybrid PyTorch model and is not a `nn.Module` type, then you must pass in the inputs to the function not to the whole model when running `.distill`.

In [ ]:
symbolic_function.switch_to_symbolic()

✅ Successfully switched callable_function to symbolic equations for all 1 dimensions:
   Dimension 0: ((sin(x4) * 3.0) + (x0 * (x0 + -6.008807e-8))) + -4.0
   Variables: ['x0', 'x4']
🎯 All 1 output dimensions now using symbolic equations.


In [24]:
# Clean up 
import os
import shutil
if os.path.exists('SR_output'):
    shutil.rmtree('SR_output')
    os.remove('model_weights.pth')

if os.path.exists('symtorch_data'):
    shutil.rmtree('symtorch_data')